In [3]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage


@tool
def add(a: int, b: int) -> int:
    """دو عدد را جمع می‌کند."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """دو عدد را ضرب می‌کند."""
    return a * b


tools = [add, multiply]

tool_map = {
    tool.name: tool
    for tool in tools
}


model = init_chat_model(
    "qwen3:1.7b",
    model_provider="ollama",
    temperature=0
)

model_with_tools = model.bind_tools(tools)


def run_agent(question: str):

    messages = [
        HumanMessage(question)
    ]

    while True:

        print("\n[LLM]")
        
        response = model_with_tools.invoke(messages)

        messages.append(response)


        # -------------------------
        # آیا Tool لازم است؟
        # -------------------------

        if not response.tool_calls:
            print("[FINAL]")
            print(response.content)
            return response.content


        # -------------------------
        # اجرای Toolها
        # -------------------------

        for tool_call in response.tool_calls:

            print(
                f"[TOOL CALL] "
                f"{tool_call['name']} "
                f"{tool_call['args']}"
            )

            selected_tool = tool_map[
                tool_call["name"]
            ]

            result = selected_tool.invoke(
                tool_call["args"]
            )

            print(
                f"[TOOL RESULT] {result}"
            )

            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )

#--------------------------
run_agent(
    "عدد 12 را در 3 ضرب کن و نتیجه را با عدد 10 جمع کن"
)


[LLM]
[TOOL CALL] multiply {'a': 12, 'b': 3}
[TOOL RESULT] 36
[TOOL CALL] add {'a': 36, 'b': 10}
[TOOL RESULT] 46

[LLM]
[FINAL]
The result of multiplying 12 by 3 is 36. Adding 10 to 36 gives:

**36 + 10 = 46**

Final answer: **46**


'The result of multiplying 12 by 3 is 36. Adding 10 to 36 gives:\n\n**36 + 10 = 46**\n\nFinal answer: **46**'